# Lecture 4.1 — RunResult in Depth: new_items, input_items, usage, last_agent

**Section 04 — Running Agents, Results & Streaming**

In this notebook we go far beyond `final_output`. Every call to `Runner.run()` hands back a `RunResult` object, and that object is carrying a lot more than the answer you print at the end. It holds the full transcript of everything that happened during the run: every tool call, every tool result, every message, and every raw model response.

By the end of this notebook you will be able to:

- Read `new_items` to see the complete list of things that happened during a run
- Filter that list by item type to pull out just the tool calls, just the tool outputs, or just the messages
- Use `ItemHelpers` to extract text without writing your own parsing code
- Use `to_input_list()` to manually chain runs together into a multi-turn conversation
- Read `last_agent`, `last_response_id`, `input`, and `raw_responses` and know what each one is for
- Cast a structured `final_output` to its real type with `final_output_as()`
- Recognize `agent_tool_invocation` metadata when an agent is invoked as a nested tool


## Cell 1 — Install the OpenAI Agents SDK

📌 **Notebook update notice:** this lecture's markdown references `openai-agents==0.18.0` as the pinned version. Since recording, a downstream dependency change (`openai>=2.45.0`, released July 9, 2026) broke `openai-agents` versions below 0.18.1 — `Runner.run()` will fail on the version stated above. This notebook has been updated to pin `openai-agents==0.18.3`, which fixes the issue without changing any of the code or concepts taught in the lecture. Please use the version pinned below, not the one mentioned in the recording.

This notebook uses the OpenAI Agents SDK for Python. The cell below installs it.

The version is pinned so that the examples in this notebook behave exactly as shown, regardless of when you're watching this. If you already have the package installed in this session, running the cell again is harmless, it just confirms the version and moves on.

In [ ]:
# Pinned for reproducibility. Updated after recording — see the
# notice above. Originally pinned to 0.18.0 as stated in the
# video; updated to 0.18.3 to fix a breaking change introduced
# by openai>=2.45.0 (July 9, 2026).
# To use the latest version instead, run: pip install openai-agents
!pip install openai-agents==0.18.3 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.7/859.7 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.4 MB/s eta 0:00:00


## Cell 2: Set Up Your OpenAI API Key

We read the API key from Colab Secrets rather than typing it into the notebook. This keeps the key out of the notebook file itself.

**Steps to add your key in Colab:**

1. Click the key icon (🔑) in the left sidebar to open the Secrets panel.
2. Click **Add new secret**.
3. Set the name to `OPENAI_API_KEY`.
4. Paste your API key as the value.
5. Toggle **Notebook access** on for this secret.

**Running locally instead of Colab?** Set the environment variable in your terminal before launching Jupyter, for example `export OPENAI_API_KEY="your-key-here"`, and skip the `userdata.get()` call below.

In [ ]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## Cell 3: Set the Model Name

We declare `MODEL_NAME` once here and reuse it in every `Agent` we build in this notebook. If you want to try a different model, change it in this one place and every agent below picks up the change automatically.

In [ ]:
# See latest models at: https://platform.openai.com/docs/models
MODEL_NAME = "gpt-5.4-mini"

## Cell 4: Imports

A quick tour of what we're importing and why:

| Import | What it's for |
|---|---|
| `BaseModel` (pydantic) | Defines the structured output type we'll use later with `final_output_as()` |
| `Reasoning` (openai.types.shared) | Configures reasoning effort on `ModelSettings` |
| `Agent`, `Runner` | Core SDK classes you've used since Section 1 |
| `ModelSettings` | Passes `reasoning` and `verbosity` settings to an agent |
| `function_tool` | Turns a plain Python function into a tool the agent can call |
| `ItemHelpers` | Utility class with `text_message_output()`, `text_message_outputs()`, and more |
| `MessageOutputItem`, `ToolCallItem`, `ToolCallOutputItem` | The `RunItem` subtypes we'll filter `new_items` by |
| `HandoffOutputItem` | A `RunItem` subtype produced when one agent hands off to another |
| `ReasoningItem` | A `RunItem` subtype produced when a reasoning model emits a reasoning trace |
| `AgentToolInvocation` | The metadata type returned by `result.agent_tool_invocation` |
| `RunResult` | The return type of `Runner.run()`, used as a type hint for our `custom_output_extractor` in Cell 14 |

Every one of these `RunItem` types and `ItemHelpers` is importable straight from the top-level `agents` package, so you never need to dig into submodules to use them.

In [ ]:
from pydantic import BaseModel
from openai.types.shared import Reasoning
from openai.types.responses import ResponseOutputText
from agents import (
    Agent,
    AgentToolInvocation,
    HandoffOutputItem,
    ItemHelpers,
    MessageOutputItem,
    ModelSettings,
    ReasoningItem,
    RunResult,
    Runner,
    ToolCallItem,
    ToolCallOutputItem,
    function_tool
)

## Cell 5: RunResult and RunResultBase — the Big Picture

`Runner.run()` returns a `RunResult`. `Runner.run_streamed()` (Lectures 4.2 and 4.3) returns a `RunResultStreaming`. Both of them inherit from `RunResultBase`, which is where most of the surfaces we'll use in this notebook actually live.

Here's the full reference table for `RunResultBase`. We won't touch every row today (`interruptions` is Update U3, `context_wrapper.usage` is Lecture 4.7), but it's worth seeing the whole shape up front.

| Property or method | Type | What it gives you |
|---|---|---|
| `final_output` | `Any` | The final answer, or `None` if the run stopped early |
| `last_agent` | `Agent` | The agent that produced the final output |
| `new_items` | `list[RunItem]` | The full run transcript |
| `input` | `str \| list` | The base input this run used |
| `to_input_list(mode=...)` | `list` | A plain-item view of the run, ready for the next turn |
| `raw_responses` | `list[ModelResponse]` | One raw response per model call in the run |
| `last_response_id` | `str \| None` | The ID of the last model call, for Responses API chaining |
| `interruptions` | `list` | Pending tool approvals (Update U3) |
| `agent_tool_invocation` | `AgentToolInvocation \| None` | Metadata when this run happened inside `Agent.as_tool()` |
| `final_output_as(cls)` | `T` | A typed view of `final_output` |
| `context_wrapper.usage` | `Usage` | Aggregated token usage for the run (Lecture 4.7) |

The key idea for this whole lecture: **`final_output` is just the tip of the iceberg.** `new_items` is the complete transcript underneath it, and everything else on this table is a different lens on that same run. Let's start generating some rich results to look at.

## Cell 6: Set Up a Tool-Using Agent

To get a `RunResult` worth exploring, we need a run that produces more than one item, a plain text answer only gives us a single `MessageOutputItem`. So we'll give this agent two tools and ask a question that needs both.

A quick note on `model_settings`: `reasoning=Reasoning(effort="none")` turns off extended reasoning traces so the output stays simple to read, and `verbosity="low"` keeps responses short. You've used this pattern since earlier sections.

In [ ]:
@function_tool
def get_weather(city: str) -> str:
    """Returns the current weather for a city.

    Args:
        city: The city to check weather for.
    """
    return f"The weather in {city} is sunny and 24°C."


@function_tool
def get_population(city: str) -> str:
    """Returns the population of a city.

    Args:
        city: The city to check population for.
    """
    return f"The population of {city} is approximately 2.1 million."


agent = Agent(
    name="City Agent",
    instructions=(
        "You are a city information assistant. "
        "Use available tools to answer questions about cities."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[get_weather, get_population],
)

result = await Runner.run(
    agent,
    "What is the weather and population of Mumbai?",
)

print("Final output:", result.final_output)

Final output: Mumbai: sunny, 24°C; population approximately 2.1 million.


## Cell 7: Explore new_items — the Full Run Transcript

`result.new_items` is a `list[RunItem]`. Every single thing the SDK produced during the run shows up here, in order: the model deciding to call a tool, the tool's return value coming back, and finally the model's text response.

Each `RunItem` carries a `.type` string literal and a `.raw_item` (the underlying Responses API object). In a run like ours, with two tool calls, you'll typically see `MessageOutputItem`, `ToolCallItem`, and `ToolCallOutputItem`. Other lecture scenarios can add `HandoffCallItem` and `HandoffOutputItem` (Section 5), `ReasoningItem` (reasoning models), or approval and MCP items (Update U2 and U3).

In [ ]:
print(f"Total items in this run: {len(result.new_items)}")
print()

for i, item in enumerate(result.new_items):
    print(
        f"Item {i}: {type(item).__name__} "
        f"(type='{item.type}')"
    )

Total items in this run: 5

Item 0: ToolCallItem (type='tool_call_item')
Item 1: ToolCallItem (type='tool_call_item')
Item 2: ToolCallOutputItem (type='tool_call_output_item')
Item 3: ToolCallOutputItem (type='tool_call_output_item')
Item 4: MessageOutputItem (type='message_output_item')


## Cell 8: Filter new_items by Type

Reading the raw list is useful once. Filtering it by type with `isinstance()` is what you'll actually reach for in production, this is the core pattern behind logs, audit trails, and any UI that wants to render tool calls differently from messages.

Two properties worth knowing on the tool items themselves:

- `ToolCallItem.tool_name`: the name of the tool the model called
- `ToolCallOutputItem.output`: the actual Python return value of your tool function (not a string representation of it)

And for the message text, we reach for `ItemHelpers.text_message_output()`, covered properly in the next cell.

In [ ]:
tool_calls = [
    item for item in result.new_items
    if isinstance(item, ToolCallItem)
]
tool_outputs = [
    item for item in result.new_items
    if isinstance(item, ToolCallOutputItem)
]
messages = [
    item for item in result.new_items
    if isinstance(item, MessageOutputItem)
]

print(f"Tool calls: {len(tool_calls)}")
for tc in tool_calls:
    print(f"  - {tc.tool_name}")

print(f"\nTool outputs: {len(tool_outputs)}")
for to in tool_outputs:
    print(f"  - {to.output}")

print(f"\nMessages: {len(messages)}")
for msg in messages:
    # Manual extraction: loop over the raw content blocks
    # yourself and pick out the text ones.
    manual_text = ""
    for content_block in msg.raw_item.content:
        if isinstance(content_block, ResponseOutputText):
            manual_text += content_block.text or ""
    print(f"  - {manual_text[:80]}")

Tool calls: 2
  - get_weather
  - get_population

Tool outputs: 2
  - The weather in Mumbai is sunny and 24°C.
  - The population of Mumbai is approximately 2.1 million.

Messages: 1
  - Mumbai: sunny, 24°C; population approximately 2.1 million.


## Cell 9: ItemHelpers — Utility Methods

`ItemHelpers` is a small class of classmethods that save you from re-writing the same extraction logic in every project. The three you'll use constantly:

| Method | Signature | What it does |
|---|---|---|
| `text_message_output(message)` | `MessageOutputItem -> str` | Extracts all text content from a single message item |
| `text_message_outputs(items)` | `list[RunItem] -> str` | Concatenates text from every `MessageOutputItem` in a list, skipping everything else |
| `input_to_new_input_list(input)` | `str \| list -> list` | Normalizes a plain string or a list into the list-of-input-items shape the SDK expects |

There are also `extract_last_content()`, `extract_last_text()`, and `extract_text()` for pulling content out of a single raw message, but the three above are the ones you'll reach for most often when working with `new_items`.

In [ ]:
for msg in messages:
    text = ItemHelpers.text_message_output(msg)
    print("Single message text:", text[:80])

all_text = ItemHelpers.text_message_outputs(result.new_items)
print("\nAll message text combined:")
print(all_text[:200])

input_list = ItemHelpers.input_to_new_input_list("Hello")
print("\nStr input normalised to list:", input_list)

Single message text: Mumbai: sunny, 24°C; population approximately 2.1 million.

All message text combined:
Mumbai: sunny, 24°C; population approximately 2.1 million.

Str input normalised to list: [{'content': 'Hello', 'role': 'user'}]


## Cell 10: to_input_list() — Chaining Runs Manually

This is the mechanism behind every manual multi-turn conversation you'll build before you learn about Sessions in Update U1. Call `result.to_input_list()`, append the user's next message, and pass the combined list straight back into `Runner.run()`. That's a complete chat loop in three lines.

`to_input_list()` takes a `mode` keyword:

- `mode="preserve_all"` (the default): converts `new_items` into the full plain-item history, every tool call, every tool output, every model response. This is what you want for a standard manual chat loop.
- `mode="normalized"`: prefers the canonical continuation input in cases where a handoff input filter rewrote the model's history. For an ordinary run like ours, with no handoffs, the two modes produce identical results.

Sessions (Update U1) will automate this pattern for you entirely, but understanding what's happening underneath it first will make Sessions click much faster when we get there.

In [ ]:
input_for_next_turn = result.to_input_list()

print(
    f"to_input_list() item count: "
    f"{len(input_for_next_turn)}"
)
print(
    f"First item role: "
    f"{input_for_next_turn[0].get('role', 'unknown')}"
)

result2 = await Runner.run(
    agent,
    input_for_next_turn + [
        {"role": "user", "content": "And what about Delhi?"}
    ],
)

print("\nContinuation result:", result2.final_output)

to_input_list() item count: 6
First item role: user

Continuation result: Delhi: sunny, 24°C; population approximately 2.1 million.


## Cell 11: last_agent and last_response_id

`last_agent` is the agent that produced the final output. In a single-agent run like this one it's always going to be the same agent you started with. Once you get to Section 5 and start building multi-agent systems with handoffs, `last_agent` becomes the thing you check to know which agent should handle the user's next message.

`last_response_id` is the response ID of the last model call in the run. It exists for one specific purpose: pass it as `previous_response_id` in `RunConfig` to chain requests through the OpenAI Responses API, letting the API reuse cached context from the previous call instead of resending it. We'll put this to use in Lecture 4.4 when we cover `RunConfig`.

In [ ]:
print("Last agent name:", result.last_agent.name)
print("Last response ID:", result.last_response_id)

Last agent name: City Agent
Last response ID: resp_00780034e14fa402006a51a6bfe5b8819cb37a406573bf6b74


## Cell 12: input and raw_responses

`result.input` is the base input this run segment actually used. In ordinary runs, that's just what you passed to `Runner.run()`. If a handoff input filter had rewritten the conversation history, this would reflect the filtered version instead, that's a Section 5 topic.

`result.raw_responses` is a list of `ModelResponse` objects, one per model call made during the run. In a tool-using run like ours there are typically two calls: the first one where the model decides to call tools, and a follow-up call after the tool results come back. Each `ModelResponse` carries its own `response_id` and `usage`, which is handy for provider-level diagnostics or per-call token tracking. We'll go deep on usage tracking in Lecture 4.7.

In [ ]:
print("Run input:", result.input)

print(
    f"\nNumber of raw model responses: "
    f"{len(result.raw_responses)}"
)
for i, resp in enumerate(result.raw_responses):
    print(
        f"  Response {i}: "
        f"id={resp.response_id}, "
        f"usage={resp.usage}"
    )

Run input: What is the weather and population of Mumbai?

Number of raw model responses: 2
  Response 0: id=resp_00780034e14fa402006a51a6be63a8819c9d01d826bc05207c, usage=Usage(requests=1, input_tokens=126, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=48, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=174, request_usage_entries=[])
  Response 1: id=resp_00780034e14fa402006a51a6bfe5b8819cb37a406573bf6b74, usage=Usage(requests=1, input_tokens=217, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=20, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=237, request_usage_entries=[])


## Cell 13: final_output_as() — a Typed View of final_output

`final_output` is typed as `Any` for a reason: handoffs can change which agent finishes a run, so the SDK has no way to statically know the full set of possible output types ahead of time.

`final_output_as(cls)` gives you a typed view of that value. By default it's purely for your type checker, it does not perform a runtime check at all. If `output_type=CityReport` is set on the agent that finishes the run, `final_output` is already a `CityReport` instance the moment `Runner.run()` returns, so calling `final_output_as(CityReport)` on it doesn't change anything observable. It just tells your editor and type checker what's already true.

To actually see `final_output_as()` do something at runtime, pass `raise_if_incorrect_type=True`. That makes it raise a `TypeError` if `final_output` is not an instance of the class you pass in. We'll deliberately cast to the wrong type to see that happen.

In [ ]:
class CityReport(BaseModel):
    city: str
    weather: str
    population: str
    summary: str


typed_agent = Agent(
    name="Typed City Agent",
    instructions=(
        "You are a city information assistant. "
        "Use tools to gather information and return a "
        "structured report."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[get_weather, get_population],
    output_type=CityReport,
)

typed_result = await Runner.run(
    typed_agent,
    "Give me a complete report on Bangalore.",
)

# No-op at runtime: report is already a CityReport.
# This cast only helps your type checker.
report = typed_result.final_output_as(CityReport)

print("City:", report.city)
print("Weather:", report.weather)
print("Population:", report.population)
print("Summary:", report.summary)

# Now force a real runtime check with raise_if_incorrect_type=True.
class WrongType(BaseModel):
    unrelated_field: str

try:
    typed_result.final_output_as(WrongType, raise_if_incorrect_type=True)
except TypeError as e:
    print("\nCaught expected error:", e)

City: Bangalore
Weather: Sunny, 24°C
Population: Approximately 2.1 million
Summary: Bangalore is currently sunny and 24°C, with an approximate population of 2.1 million.

Caught expected error: Final output is not of type WrongType


## Cell 14: agent_tool_invocation — Nested as_tool() Metadata

`Agent.as_tool()` lets you use one agent as a tool inside another agent's toolset, you first saw this pattern in Lecture 3.4. When the outer agent invokes that nested agent, the nested run's `RunResult` carries `agent_tool_invocation`, an `AgentToolInvocation` with three fields:

- `tool_name`
- `tool_call_id`
- `tool_arguments` (the raw JSON arguments string)

For an ordinary top-level run, `agent_tool_invocation` is always `None`, that includes `outer_result` below. It only becomes populated on the **nested** run's `RunResult`, and the only place the SDK actually hands you that nested `RunResult` is inside a `custom_output_extractor` passed to `as_tool()`.

So to actually see a populated `agent_tool_invocation`, we supply a `custom_output_extractor`. The SDK calls it with the nested run's `RunResult` before the tool's output goes back to the outer agent, so we can inspect and print `agent_tool_invocation` right there.

In [ ]:
async def extract_and_log_invocation(nested_result: RunResult) -> str:
    invocation = nested_result.agent_tool_invocation
    print("Nested agent_tool_invocation:")
    print("  tool_name:", invocation.tool_name)
    print("  tool_call_id:", invocation.tool_call_id)
    print("  tool_arguments:", invocation.tool_arguments)
    return nested_result.final_output


translator = Agent(
    name="Translator",
    instructions="You translate text to Spanish.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

orchestrator = Agent(
    name="Orchestrator",
    instructions=(
        "Use the translate_to_spanish tool when asked "
        "to translate."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[
        translator.as_tool(
            tool_name="translate_to_spanish",
            tool_description="Translate text to Spanish.",
            custom_output_extractor=extract_and_log_invocation,
        )
    ],
)

outer_result = await Runner.run(
    orchestrator,
    "Translate 'Good morning' to Spanish.",
)

print("\nOuter result:", outer_result.final_output)
print(
    "Outer agent_tool_invocation:",
    outer_result.agent_tool_invocation,
)

Nested agent_tool_invocation:
  tool_name: translate_to_spanish
  tool_call_id: call_h7KztfZeLb5YVGs36jFAt6R8
  tool_arguments: {"input":"Good morning"}

Outer result: Buenos días
Outer agent_tool_invocation: None


## Cell 15: RunResult Surface Reference

A single table to keep handy as you build with `RunResult` going forward:

| Property / Method | Type | What it gives you |
|---|---|---|
| `final_output` | `Any` | Final answer, or `None` |
| `last_agent` | `Agent` | Agent that produced the final output |
| `new_items` | `list[RunItem]` | Full run transcript |
| `input` | `str \| list` | Base input this run used |
| `to_input_list(mode=...)` | `list` | Next-turn input list |
| `raw_responses` | `list[ModelResponse]` | Raw model calls |
| `last_response_id` | `str \| None` | Responses API chain ID |
| `interruptions` | `list` | Pending approvals (Update U3) |
| `agent_tool_invocation` | `AgentToolInvocation \| None` | Nested `as_tool()` metadata |
| `final_output_as(cls)` | `T` | Typed view of `final_output` |
| `context_wrapper.usage` | `Usage` | Aggregated token usage (Lecture 4.7) |

That's the full shape of `RunResult`. In the next lecture we switch from inspecting a finished run to watching one happen in real time, with `Runner.run_streamed()` and async event streaming.